# CF Rating Engine — Validation & Analysis

Data source: `data/validation/deflation/per_participant.csv` (all cached contests).

Stage definitions — each new-system account plays up to 6 ramp-up contests before becoming "old":

| Label | Contest # | trueOld offset | CSV key |
|-------|-----------|---------------|---------|
| s1 | 1st | +1400 | stage 0 |
| s2 | 2nd | +900  | stage 1 |
| s3 | 3rd | +550  | stage 2 |
| s4 | 4th | +300  | stage 3 |
| s5 | 5th | +150  | stage 4 |
| s6 | 6th | +50   | stage 5 |
| old | 7th+ | +0  | stage 7 |

## General Accuracy (all cached contests)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from utils import PROJECT_ROOT

df = pd.read_csv(str(PROJECT_ROOT) + "/data/validation/deflation/per_participant.csv")
df["stage"] = df["stage"].astype(int)
df["abs_error"] = df["error"].abs()
df["exact"]    = df["abs_error"] == 0
df["within_1"] = df["abs_error"] <= 1
df["within_5"] = df["abs_error"] <= 5

df["rating_bin"] = pd.cut(df["old_rating"], bins=range(0, 4200, 200))

mean_err = df.groupby("rating_bin", observed=True)["error"].mean()
median_err = df.groupby("rating_bin", observed=True)["error"].median()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

bars = ax1.bar(range(len(mean_err)), mean_err.values)
ax1.set_xticks(range(len(mean_err)))
ax1.set_xticklabels(mean_err.index, rotation=45, ha="right")
ax1.axhline(0, color="black", linewidth=0.8)
ax1.set_title("Mean Error by Rating Bucket")
ax1.set_ylabel("Mean Error")
for bar, val in zip(bars, mean_err.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f"{val:.2f}", ha="center", va="bottom" if val >= 0 else "top", fontsize=8)

bars = ax2.bar(range(len(median_err)), median_err.values)
ax2.set_xticks(range(len(median_err)))
ax2.set_xticklabels(median_err.index, rotation=45, ha="right")
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_title("Median Error by Rating Bucket")
ax2.set_ylabel("Median Error")
for bar, val in zip(bars, median_err.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f"{val:.2f}", ha="center", va="bottom" if val >= 0 else "top", fontsize=8)

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.hist(df["old_rating"], bins=range(0, 4200, 200), edgecolor="black", linewidth=0.5)
ax.set_yscale("log")
ax.set_xlabel("Old Rating")
ax.set_ylabel("Participant Count (log scale)")
ax.set_title("Rating Distribution of All Participants")
ax.set_xticks(range(0, 4200, 200))
ax.set_xticklabels(range(0, 4200, 200), rotation=45, ha="right")

fig.tight_layout()
plt.show()


In [ ]:
log_bins = [0, 5, 10, 25, 50, 100, 200, 500, 1000, 2000, 5000, 10000, 50000]
rank_log_bins = pd.cut(df["rank"], bins=log_bins)

mean_err_log   = df.groupby(rank_log_bins, observed=True)["error"].mean()
median_err_log = df.groupby(rank_log_bins, observed=True)["error"].median()
mean_abs_log   = df.groupby(rank_log_bins, observed=True)["abs_error"].mean()
median_abs_log = df.groupby(rank_log_bins, observed=True)["abs_error"].median()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, data, title in zip(
    axes.flat,
    [mean_err_log, median_err_log, mean_abs_log, median_abs_log],
    ["Mean Error by Rank", "Median Error by Rank", "Mean Abs Error by Rank", "Median Abs Error by Rank"],
):
    bars = ax.bar(range(len(data)), data.values)
    ax.set_xticks(range(len(data)))
    ax.set_xticklabels(data.index, rotation=45, ha="right", fontsize=8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(title)
    ax.set_ylabel(title.split(" by ")[0])
    for bar, val in zip(bars, data.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{val:.2f}",
            ha="center",
            va="bottom" if val >= 0 else "top",
            fontsize=8,
        )

fig.tight_layout()
plt.show()


In [ ]:
contest_acc = (
    df.groupby("contest_id")[["exact", "within_1", "within_5"]]
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .sort_values("contest_id")
)

fig, ax = plt.subplots(figsize=(max(10, len(contest_acc) * 0.5), 5))

x = range(len(contest_acc))
w = 0.28

def labeled_bars(ax, positions, values, width, label, color):
    bars = ax.bar(positions, values, width=width, label=label, color=color)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{val:.1f}", ha="center", va="bottom", fontsize=6, rotation=90)

labeled_bars(ax, [i - w for i in x], contest_acc["exact"],    w, "Exact",     "steelblue")
labeled_bars(ax, [i     for i in x], contest_acc["within_1"], w, "Within ±1", "seagreen")
labeled_bars(ax, [i + w for i in x], contest_acc["within_5"], w, "Within ±5", "tomato")

ax.set_xticks(list(x))
ax.set_xticklabels(contest_acc["contest_id"], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("% of Participants")
ax.set_title("Accuracy per Contest (Exact / ±1 / ±5)")
ax.legend()
ax.set_ylim(0, 115)

fig.tight_layout()
plt.show()

print(contest_acc.to_string(index=False))


# Deflation Analysis — Old vs New Era Contests

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from utils import PROJECT_ROOT

summary = pd.read_csv(str(PROJECT_ROOT) + "/data/validation/deflation/summary.csv")

if "df" not in dir():
    pp = pd.read_csv(str(PROJECT_ROOT) + "/data/validation/deflation/per_participant.csv")
    pp["stage"] = pp["stage"].astype(int)
else:
    pp = df

display(summary[["contest_id", "era", "n", "new_sys_frac",
                 "mean_actual", "mean_pred", "mean_err"]].round(3))

In [ ]:
## Cell 2 — Mean actual vs predicted delta per contest
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Mean Delta per Contest: CF actual vs Our Prediction", fontsize=13, fontweight="bold")

for ax, era, title in zip(axes, ["old", "new"], ["Old-era contests (1000–1585)", "New-era contests (>2000)"]):
    sub = summary[summary["era"] == era].sort_values("contest_id")
    x   = np.arange(len(sub))
    w   = 0.35
    bars_a = ax.bar(x - w/2, sub["mean_actual"], w, label="CF actual",    color="steelblue")
    bars_p = ax.bar(x + w/2, sub["mean_pred"],   w, label="Our engine",   color="tomato", alpha=0.8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["contest_id"], rotation=45, ha="right")
    ax.set_ylabel("Mean delta per player")
    ax.set_title(title)
    ax.legend()
    for bar in bars_a:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 1.5,
                f"{bar.get_height():.1f}", ha="center", va="top", fontsize=7, color="white")
    for bar in bars_p:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=7)

fig.tight_layout()
plt.show()

In [ ]:
## Cell 3 — Mean error vs new-sys fraction (scatter + bar)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Prediction Error vs New-System Account Fraction", fontsize=13, fontweight="bold")

colors = {"old": "steelblue", "new": "tomato"}
for _, row in summary.iterrows():
    ax1.scatter(row["new_sys_frac"] * 100, row["mean_err"],
                color=colors[row["era"]], s=80, zorder=3)
    ax1.annotate(str(int(row["contest_id"])),
                 (row["new_sys_frac"] * 100, row["mean_err"]),
                 textcoords="offset points", xytext=(4, 3), fontsize=7)

ax1.axhline(0, color="black", linewidth=0.8)
ax1.set_xlabel("New-system accounts (% of contest)")
ax1.set_ylabel("Mean error (pred − actual) per player")
ax1.set_title("Error vs New-Sys Fraction")
from matplotlib.patches import Patch
ax1.legend(handles=[Patch(color="steelblue", label="old era"),
                    Patch(color="tomato",     label="new era")])

# Bar: mean_err per contest sorted by new_sys_frac
sub_sorted = summary.sort_values("new_sys_frac")
bar_colors = [colors[e] for e in sub_sorted["era"]]
bars = ax2.bar(range(len(sub_sorted)), sub_sorted["mean_err"], color=bar_colors)
ax2.set_xticks(range(len(sub_sorted)))
ax2.set_xticklabels(
    [f"{int(r['contest_id'])}\n({100*r['new_sys_frac']:.0f}%)" for _, r in sub_sorted.iterrows()],
    rotation=45, ha="right", fontsize=8)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_ylabel("Mean error per player")
ax2.set_title("Error per Contest (sorted by new-sys %)")
for bar, val in zip(bars, sub_sorted["mean_err"]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{val:+.1f}", ha="center", va="bottom", fontsize=8)

fig.tight_layout()
plt.show()

In [ ]:
## Cell 5 — Error distribution boxplot per contest (from per_participant cache)
contests_ordered = summary.sort_values("new_sys_frac")["contest_id"].tolist()
fig, ax = plt.subplots(figsize=(16, 5))
fig.suptitle("Error Distribution per Contest (sorted by new-sys %)", fontsize=13, fontweight="bold")

data_by_contest = [
    pp[pp["contest_id"] == cid]["error"].dropna().values
    for cid in contests_ordered
]
bp = ax.boxplot(data_by_contest, patch_artist=True, showfliers=False,
                medianprops=dict(color="black", linewidth=1.5))

era_map = dict(zip(summary["contest_id"], summary["era"]))
frac_map = dict(zip(summary["contest_id"], summary["new_sys_frac"]))
box_colors = {"old": "lightsteelblue", "new": "lightsalmon"}
for patch, cid in zip(bp["boxes"], contests_ordered):
    patch.set_facecolor(box_colors[era_map[cid]])

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xticks(range(1, len(contests_ordered) + 1))
ax.set_xticklabels(
    [f"{int(c)}\n({100*frac_map[c]:.0f}%)" for c in contests_ordered],
    rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Error (pred − actual)")
ax.set_title("IQR of prediction error per contest")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="lightsteelblue", label="old era"),
                   Patch(color="lightsalmon",    label="new era")])

fig.tight_layout()
plt.show()

# Stage-Level Analysis

s1–s6 = six ramp-up contests for new CF accounts; **old** = fully established (offset 0).
All graphs below use `data/validation/deflation/` which covers 12 selected contests.

In [ ]:
## Pair 1 — Participant distribution by stage per contest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import PROJECT_ROOT

if "summary" not in dir():
    summary = pd.read_csv(str(PROJECT_ROOT) + "/data/validation/deflation/summary.csv")

STAGE_KEYS   = [0, 1, 2, 3, 4, 5, 7]
STAGE_LABELS = {0: "s1", 1: "s2", 2: "s3", 3: "s4", 4: "s5", 5: "s6", 7: "old"}
STAGE_COLS   = {k: (f"stage{k}" if k < 7 else "old") for k in STAGE_KEYS}
LABEL_ORDER  = [STAGE_LABELS[k] for k in STAGE_KEYS]
COLORS       = plt.cm.Set2(np.linspace(0, 1, len(STAGE_KEYS)))

cnt = summary.set_index("contest_id")
all_cids = summary["contest_id"].tolist()

# Absolute count table
count_df = pd.DataFrame(index=all_cids)
count_df.index.name = "contest"
count_df["n"] = cnt["n"].values
for k in STAGE_KEYS:
    count_df[STAGE_LABELS[k]] = cnt[f"n_{STAGE_COLS[k]}"].astype(int).values

# Percentage table
pct_df = count_df[LABEL_ORDER].div(count_df["n"], axis=0).mul(100)

# Combined display table: count + % side by side
table = count_df[["n"]].copy()
for lbl in LABEL_ORDER:
    table[f"{lbl}"] = count_df[lbl]
    table[f"{lbl}_%"] = pct_df[lbl].round(1)
display(table)

# Charts
xtick_labels = [f"{c}\n({cnt.loc[c,'era']})" for c in all_cids]
x = np.arange(len(all_cids))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Participant Distribution by Stage per Contest", fontsize=13, fontweight="bold")

# Stacked % bar
bottoms = np.zeros(len(all_cids))
for i, lbl in enumerate(LABEL_ORDER):
    vals = pct_df[lbl].values
    ax1.bar(x, vals, bottom=bottoms, color=COLORS[i], label=lbl)
    for xi, v, b in zip(x, vals, bottoms):
        if v > 4:
            ax1.text(xi, b + v / 2, f"{v:.0f}%", ha="center", va="center",
                     fontsize=7, color="white", fontweight="bold")
    bottoms += vals
ax1.set_xticks(x); ax1.set_xticklabels(xtick_labels, rotation=45, ha="right", fontsize=8)
ax1.set_ylabel("% of participants"); ax1.set_title("Stage composition (%)")
ax1.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8); ax1.set_ylim(0, 115)

# Stacked absolute counts
bottoms = np.zeros(len(all_cids))
for i, lbl in enumerate(LABEL_ORDER):
    vals = count_df[lbl].values / 1000
    ax2.bar(x, vals, bottom=bottoms, color=COLORS[i], label=lbl)
    bottoms += vals
ax2.set_xticks(x); ax2.set_xticklabels(xtick_labels, rotation=45, ha="right", fontsize=8)
ax2.set_ylabel("Player count (thousands)"); ax2.set_title("Absolute stage counts")
ax2.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

fig.tight_layout()
plt.show()

In [ ]:
## Pair 2 — Mean prediction error per stage per contest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import PROJECT_ROOT

if "summary" not in dir():
    summary = pd.read_csv(str(PROJECT_ROOT) + "/data/validation/deflation/summary.csv")

STAGE_KEYS   = [0, 1, 2, 3, 4, 5, 7]
STAGE_LABELS = {0: "s1", 1: "s2", 2: "s3", 3: "s4", 4: "s5", 5: "s6", 7: "old"}
STAGE_COLS   = {k: (f"stage{k}" if k < 7 else "old") for k in STAGE_KEYS}
LABEL_ORDER  = [STAGE_LABELS[k] for k in STAGE_KEYS]
COLORS       = plt.cm.Set2(np.linspace(0, 1, len(STAGE_KEYS)))

cnt = summary.set_index("contest_id")
all_cids = summary["contest_id"].tolist()

# Error table
err_df = pd.DataFrame(index=all_cids)
err_df.index.name = "contest"
for k in STAGE_KEYS:
    col = f"mean_err_{STAGE_COLS[k]}"
    err_df[STAGE_LABELS[k]] = pd.to_numeric(cnt[col], errors="coerce").values
display(err_df.round(2))

# Grouped bar chart
x = np.arange(len(all_cids))
w = 0.11
xtick_labels = [f"{c}\n({cnt.loc[c,'era']})" for c in all_cids]

fig, ax = plt.subplots(figsize=(16, 5))
for i, lbl in enumerate(LABEL_ORDER):
    vals = err_df[lbl].values.astype(float)
    bars = ax.bar(x + (i - 3) * w, vals, w, color=COLORS[i], label=lbl, alpha=0.85)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(xtick_labels, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Mean error (pred − actual)")
ax.set_title("Mean Prediction Error by Stage per Contest", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

In [ ]:
## Cell 6 — Aggregate mean error per stage with model fits
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import PROJECT_ROOT

# Load deflation data if not already in scope
if "pp" not in dir() or "deflation" not in str(pp.columns.tolist()):
    summary = pd.read_csv(str(PROJECT_ROOT) + "/data/validation/deflation/summary.csv")
    pp      = pd.read_csv(str(PROJECT_ROOT) + "/data/validation/deflation/per_participant.csv")
    pp["stage"] = pp["stage"].astype(int)

NEW_SYS_CONTESTS = {1400, 1585, 2000, 2050, 2130, 2200, 2234}
OFFSETS = {0: 1400, 1: 900, 2: 550, 3: 300, 4: 150, 5: 50, 7: 0}
STAGE_KEYS = [0, 1, 2, 3, 4, 5, 7]
STAGE_LABELS = {0: "s1", 1: "s2", 2: "s3", 3: "s4", 4: "s5", 5: "s6", 7: "old"}

pp_new = pp[pp["contest_id"].isin(NEW_SYS_CONTESTS)].copy()
agg = (
    pp_new.groupby("stage")["error"]
    .agg(mean="mean", median="median", n="count")
    .reindex(STAGE_KEYS)
    .reset_index()
)

old_mean = agg.loc[agg["stage"] == 7, "mean"].values[0]
agg["linear"] = old_mean * (1 - agg["stage"].map(OFFSETS) / 1400)
agg["sqrt"]   = old_mean * np.sqrt(1 - agg["stage"].map(OFFSETS) / 1400)
agg["label"]  = agg["stage"].map(STAGE_LABELS)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Mean Prediction Error by Ramp-up Stage (new-sys contests only)", fontsize=13, fontweight="bold")

x = np.arange(len(agg))
ax1.bar(x, agg["mean"], color="steelblue", alpha=0.7, label="Actual mean error")
ax1.plot(x, agg["linear"], "o--", color="tomato",   linewidth=1.5, label="Linear  C·(1−off/1400)")
ax1.plot(x, agg["sqrt"],   "s--", color="seagreen", linewidth=1.5, label="Sqrt  C·√(1−off/1400)")
ax1.axhline(0, color="black", linewidth=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels(
    [f"{STAGE_LABELS[k]}\n(+{OFFSETS[k]})" for k in STAGE_KEYS],
    rotation=0, ha="center", fontsize=9)
ax1.set_ylabel("Mean error (pred − actual)")
ax1.set_title("Error curve across stages")
ax1.legend(fontsize=8)
for xi, (_, row) in zip(x, agg.iterrows()):
    ax1.text(xi, row["mean"] + 0.2, f"{row['mean']:+.2f}", ha="center", va="bottom", fontsize=8)

ax2.bar(x, agg["n"] / 1000, color="slategray", alpha=0.7)
ax2.set_xticks(x)
ax2.set_xticklabels(
    [f"{STAGE_LABELS[k]}\n(+{OFFSETS[k]})" for k in STAGE_KEYS],
    rotation=0, ha="center", fontsize=9)
ax2.set_ylabel("Participant count (thousands)")
ax2.set_title("Sample size per stage")
for xi, (_, row) in zip(x, agg.iterrows()):
    ax2.text(xi, row["n"] / 1000 + 0.3, f"{int(row['n']/1000)}k", ha="center", va="bottom", fontsize=8)

fig.tight_layout()
plt.show()

display(agg[["label", "stage", "n", "mean", "median", "linear", "sqrt"]].rename(
    columns={"label": "stage_label", "mean": "mean_err", "median": "median_err",
             "linear": "linear_pred", "sqrt": "sqrt_pred"}).round(3))

In [ ]:
## Cell 6b — Curve fit: error = C * (1 - offset/1400)^alpha
import numpy as np
import matplotlib.pyplot as plt

# Aggregate mean error per stage (new-sys contests)
# agg must be defined from Cell 6; if not, re-derive here
if "agg" not in dir():
    import pandas as pd
    from utils import PROJECT_ROOT
    pp      = pd.read_csv(str(PROJECT_ROOT) + "/data/validation/deflation/per_participant.csv")
    pp["stage"] = pp["stage"].astype(int)
    NEW_SYS_CONTESTS = {1400, 1585, 2000, 2050, 2130, 2200, 2234}
    OFFSETS      = {0: 1400, 1: 900, 2: 550, 3: 300, 4: 150, 5: 50, 7: 0}
    STAGE_KEYS   = [0, 1, 2, 3, 4, 5, 7]
    STAGE_LABELS = {0: "s1", 1: "s2", 2: "s3", 3: "s4", 4: "s5", 5: "s6", 7: "old"}
    pp_new = pp[pp["contest_id"].isin(NEW_SYS_CONTESTS)]
    agg = (pp_new.groupby("stage")["error"].agg(mean="mean", median="median", n="count")
           .reindex(STAGE_KEYS).reset_index())
    agg["label"] = agg["stage"].map(STAGE_LABELS)

offsets = np.array([OFFSETS[k] for k in STAGE_KEYS], dtype=float)
errors  = agg["mean"].values.copy()
errors[0] = 0.0          # treat stage-0 as exact anchor

C = errors[-1]           # old-sys error = free end of the curve
x = 1 - offsets / 1400   # deflection fraction: 0 at s1, 1 at old

# Fit: log(error/C) = alpha * log(x)  →  alpha via OLS on log-log
mask = x > 0
lx = np.log(x[mask])
ly = np.log(errors[mask] / C)
alpha = float(np.dot(lx, ly) / np.dot(lx, lx))

pred = np.where(x > 0, C * x**alpha, 0.0)
ss_res   = np.sum((errors - pred) ** 2)
ss_total = np.sum((errors - errors.mean()) ** 2)
r2 = 1 - ss_res / ss_total

print(f"Fitted model:  error(offset) = {C:.3f} × (1 − offset/1400)^{alpha:.4f}")
print(f"R² = {r2:.4f}   RMSE = {np.sqrt(ss_res/len(errors)):.3f}")
print()
print(f"{'stage':>6}  {'offset':>6}  {'actual':>7}  {'fitted':>7}  {'residual':>9}")
for k, off, act, fit in zip(STAGE_KEYS, offsets, errors, pred):
    print(f"  {STAGE_LABELS[k]:>4}  {int(off):>6}  {act:>7.3f}  {fit:>7.3f}  {act-fit:>+9.3f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Fitted model: error = {C:.2f} · (1 − offset/1400)^{alpha:.3f}  (R²={r2:.4f})",
             fontsize=12, fontweight="bold")

off_dense = np.linspace(0, 1400, 300)
x_dense   = 1 - off_dense / 1400
fit_dense = np.where(x_dense > 0, C * x_dense**alpha, 0.0)

# Plot vs offset
ax1.scatter(offsets, errors, s=80, zorder=5, color="steelblue", label="Actual mean error")
ax1.plot(off_dense, fit_dense, color="tomato", linewidth=2,
         label=f"C·(1−off/1400)^{alpha:.3f}")
ax1.invert_xaxis()
ax1.set_xlabel("Ramp-up offset")
ax1.set_ylabel("Mean error (pred − actual)")
ax1.set_title("Error vs offset (x-axis inverted: s1→old)")
ax1.set_xticks(list(offsets.astype(int)))
ax1.set_xticklabels([f"{STAGE_LABELS[k]}\n({int(o)})" for k, o in zip(STAGE_KEYS, offsets)],
                    fontsize=8)
ax1.legend()
for k, off, act in zip(STAGE_KEYS, offsets, errors):
    ax1.annotate(f"{act:.2f}", (off, act), textcoords="offset points",
                 xytext=(0, 8), ha="center", fontsize=8)

# Log-log plot (shows linearity = power law fit quality)
ax2.scatter(np.log(x[mask]), np.log(errors[mask]/C), s=80, color="steelblue", zorder=5,
            label="log(error/C) vs log(1−off/1400)")
lx_dense = np.linspace(np.log(x[mask]).min() - 0.1, 0, 100)
ax2.plot(lx_dense, alpha * lx_dense, color="tomato", linewidth=2,
         label=f"slope = {alpha:.3f}")
ax2.set_xlabel("log(1 − offset/1400)")
ax2.set_ylabel("log(error / C)")
ax2.set_title("Log-log fit (linearity = power law)")
ax2.legend()

fig.tight_layout()
plt.show()

In [ ]:
## Cell 6c — Test model against each individual contest
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import PROJECT_ROOT

if "pp" not in dir():
    pp = pd.read_csv(str(PROJECT_ROOT) + "/data/validation/deflation/per_participant.csv")
    pp["stage"] = pp["stage"].astype(int)

NEW_SYS_CONTESTS = [1400, 1585, 2000, 2050, 2130, 2200, 2234]
OFFSETS      = {0: 1400, 1: 900, 2: 550, 3: 300, 4: 150, 5: 50, 7: 0}
STAGE_KEYS   = [0, 1, 2, 3, 4, 5, 7]
STAGE_LABELS = {0: "s1", 1: "s2", 2: "s3", 3: "s4", 4: "s5", 5: "s6", 7: "old"}
GLOBAL_ALPHA = 0.5955

def fit_alpha(C, offsets_arr, errors_arr):
    x = 1 - offsets_arr / 1400
    mask = (x > 0) & (errors_arr > 0)
    if mask.sum() < 2:
        return np.nan
    lx, ly = np.log(x[mask]), np.log(errors_arr[mask] / C)
    return float(np.dot(lx, ly) / np.dot(lx, lx))

# Per-contest stage means and fitted alphas
results = {}
for cid in NEW_SYS_CONTESTS:
    sub = pp[pp["contest_id"] == cid]
    stage_means = sub.groupby("stage")["error"].mean().reindex(STAGE_KEYS)
    stage_means.iloc[0] = 0.0   # anchor s1 at 0

    C = stage_means.iloc[-1]    # old-sys error for this contest
    off_arr = np.array([OFFSETS[k] for k in STAGE_KEYS], float)
    err_arr = stage_means.values.astype(float)

    alpha_fit = fit_alpha(C, off_arr, err_arr)
    x = 1 - off_arr / 1400
    pred_global = np.where(x > 0, C * x**GLOBAL_ALPHA, 0.0)
    pred_fit    = np.where(x > 0, C * x**alpha_fit,   0.0) if not np.isnan(alpha_fit) else pred_global

    valid = ~np.isnan(err_arr)
    rmse_global = np.sqrt(np.mean((err_arr[valid] - pred_global[valid])**2))
    rmse_fit    = np.sqrt(np.mean((err_arr[valid] - pred_fit[valid])**2))

    results[cid] = dict(C=C, alpha_fit=alpha_fit, stage_means=stage_means,
                        pred_global=pred_global, pred_fit=pred_fit,
                        rmse_global=rmse_global, rmse_fit=rmse_fit)

# Summary table
print(f"{'contest':>8}  {'C (old err)':>11}  {'alpha_fit':>9}  {'RMSE(global)':>13}  {'RMSE(fit)':>10}")
print("-" * 60)
for cid, r in results.items():
    print(f"{cid:>8}  {r['C']:>11.2f}  {r['alpha_fit']:>9.4f}  {r['rmse_global']:>13.3f}  {r['rmse_fit']:>10.3f}")
print(f"\nGlobal alpha = {GLOBAL_ALPHA:.4f}  (fitted on aggregate across all contests)")

# Small-multiple plots: actual vs global vs per-contest fit
n_cols = 4
n_rows = (len(NEW_SYS_CONTESTS) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows), sharey=False)
axes = axes.flat

x_ticks = list(range(len(STAGE_KEYS)))
x_labels = [STAGE_LABELS[k] for k in STAGE_KEYS]

for ax, cid in zip(axes, NEW_SYS_CONTESTS):
    r = results[cid]
    actual = r["stage_means"].values.astype(float)
    ax.bar(x_ticks, actual, color="steelblue", alpha=0.6, label="Actual")
    ax.plot(x_ticks, r["pred_global"], "o--", color="tomato",   linewidth=1.5,
            label=f"Global α={GLOBAL_ALPHA:.2f}")
    ax.plot(x_ticks, r["pred_fit"],    "s--", color="seagreen", linewidth=1.5,
            label=f"Fit α={r['alpha_fit']:.2f}")
    ax.axhline(0, color="black", linewidth=0.6)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, fontsize=8)
    ax.set_title(f"Contest {cid}  (C={r['C']:.1f})\n"
                 f"RMSE: global={r['rmse_global']:.2f}  fit={r['rmse_fit']:.2f}", fontsize=9)
    ax.set_ylabel("Mean error")
    ax.legend(fontsize=7)

for ax in list(axes)[len(NEW_SYS_CONTESTS):]:
    ax.set_visible(False)

fig.suptitle("Model vs Actual per Contest:  error = C · (1 − offset/1400)^α",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
## Cell 7 — Per-contest heatmap of mean error by stage
STAGE_KEYS   = [0, 1, 2, 3, 4, 5, 7]
STAGE_LABELS = {0: "s1", 1: "s2", 2: "s3", 3: "s4", 4: "s5", 5: "s6", 7: "old"}

contests_with_new = sorted(NEW_SYS_CONTESTS)
stage_contest_mean = (
    pp_new.groupby(["contest_id", "stage"])["error"]
    .mean()
    .unstack("stage")
    .reindex(columns=STAGE_KEYS)
    .reindex(contests_with_new)
)
stage_contest_mean.columns = [STAGE_LABELS[k] for k in STAGE_KEYS]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Mean Error per Stage, per Contest", fontsize=13, fontweight="bold")

# Heatmap
import matplotlib.colors as mcolors
vmax = stage_contest_mean.max().max()
vmin = stage_contest_mean.min().min()
norm = mcolors.TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
im = axes[0].imshow(stage_contest_mean.values, aspect="auto", cmap="RdYlGn_r", norm=norm)
axes[0].set_xticks(range(len(stage_contest_mean.columns)))
axes[0].set_xticklabels(stage_contest_mean.columns)
axes[0].set_yticks(range(len(contests_with_new)))
axes[0].set_yticklabels(contests_with_new)
axes[0].set_xlabel("Stage")
axes[0].set_ylabel("Contest")
axes[0].set_title("Heatmap (green=low error, red=high)")
plt.colorbar(im, ax=axes[0], label="Mean error")
for i, contest in enumerate(contests_with_new):
    for j, col in enumerate(stage_contest_mean.columns):
        val = stage_contest_mean.loc[contest, col]
        if not np.isnan(val):
            axes[0].text(j, i, f"{val:+.1f}", ha="center", va="center", fontsize=7,
                         color="black" if abs(val) < vmax * 0.6 else "white")

# Line plot: each contest as a line across stages
for cid in contests_with_new:
    row = stage_contest_mean.loc[cid]
    axes[1].plot(range(len(row)), row.values, "o-", label=str(cid), linewidth=1.2, markersize=4)
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_xticks(range(len(stage_contest_mean.columns)))
axes[1].set_xticklabels(stage_contest_mean.columns)
axes[1].set_ylabel("Mean error (pred − actual)")
axes[1].set_title("Error profile per contest")
axes[1].legend(title="contest", fontsize=8, title_fontsize=8)

fig.tight_layout()
plt.show()

In [ ]:
## Old-sys accounts in contest 1585 — error vs old rating
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import PROJECT_ROOT

if "df" not in dir():
    df = pd.read_csv(str(PROJECT_ROOT) + "/data/validation/deflation/per_participant.csv")
    df["stage"] = df["stage"].astype(int)

sub = df[(df["contest_id"] == 1585) & (df["stage"] == 7)].copy()

fig, ax = plt.subplots(figsize=(14, 5))

ax.scatter(sub["old_rating"], sub["error"], alpha=0.15, s=8, color="steelblue", rasterized=True)

# Rolling mean trend
sub_sorted = sub.sort_values("old_rating")
roll = sub_sorted.set_index("old_rating")["error"].rolling(200, center=True, min_periods=20).mean()
ax.plot(roll.index, roll.values, color="tomato", linewidth=2, label="Rolling mean (w=200)")

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Old rating")
ax.set_ylabel("Error (pred − actual)")
ax.set_title("Prediction Error vs Old Rating — old-sys accounts, contest 1585")
ax.legend()
print(f"n={len(sub)},  mean error={sub['error'].mean():+.2f},  median={sub['error'].median():+.2f}")
fig.tight_layout()
plt.show()

In [ ]:
## Old-sys accounts in contest 1585 — error vs rank
fig, ax = plt.subplots(figsize=(14, 5))

sub_sorted = sub.sort_values("rank")
ax.scatter(sub_sorted["rank"], sub_sorted["error"], alpha=0.15, s=8, color="steelblue", rasterized=True)

roll = sub_sorted.set_index("rank")["error"].rolling(200, center=True, min_periods=20).mean()
ax.plot(roll.index, roll.values, color="tomato", linewidth=2, label="Rolling mean (w=200)")

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Rank")
ax.set_ylabel("Error (pred − actual)")
ax.set_title("Prediction Error vs Rank — old-sys accounts, contest 1585")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
## Old-sys accounts in contest 1585 — error vs (old_rating + raw_delta)
naive = pd.read_csv(str(PROJECT_ROOT) + "/data/validation/naive/per_participant.csv")
naive_1585 = naive[naive["contest_id"] == 1585][["handle", "pred_delta_raw"]]

sub_raw = sub.merge(naive_1585, on="handle", how="inner")
sub_raw["perf_raw"] = sub_raw["old_rating"] + sub_raw["pred_delta_raw"]

fig, ax = plt.subplots(figsize=(14, 5))

sub_raw_sorted = sub_raw.sort_values("perf_raw")
ax.scatter(sub_raw_sorted["perf_raw"], sub_raw_sorted["error"],
           alpha=0.15, s=8, color="steelblue", rasterized=True)

roll = sub_raw_sorted.set_index("perf_raw")["error"].rolling(200, center=True, min_periods=20).mean()
ax.plot(roll.index, roll.values, color="tomato", linewidth=2, label="Rolling mean (w=200)")

ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("old_rating + raw_delta  (pre-correction performance)")
ax.set_ylabel("Error (pred − actual)")
ax.set_title("Prediction Error vs Pre-correction Performance — old-sys accounts, contest 1585")
ax.legend()
print(f"n={len(sub_raw)}")
fig.tight_layout()
plt.show()